# Create 256x256 crop dataset (fixed coordinate), and save them

In [34]:
import os
import numpy as np
import pandas as pd
import skimage
from tqdm import tqdm

# Training Data

In [80]:
path = '/scr/yren/micronuclei-detection/baselines/all_data_micronuclei_no_rescale/train/'
save_folder = '/scr/yren/micronuclei-detection/baselines/microSAM_finetune_data/train/'
files = os.listdir(path)
files = [file for file in files if file.endswith('.phenotype.tif')]
files = [file for file in files if not file.startswith('.')]
print(len(files))

PS = 256
for i in tqdm(range(len(files))):
    im = skimage.io.imread(path + files[i])
    gt = skimage.io.imread(path + files[i].replace('.phenotype.tif', '.phenotype_outlines.png'))

    H,W = im.shape
    patches_per_image = (W // PS) * (H // PS)
    X = np.linspace(0, W - W % PS, W // PS + 1)
    Y = np.linspace(0, H - H % PS, H // PS + 1)
    X,Y = np.meshgrid(X[:-1],Y[:-1], indexing='ij')
    X = X.reshape((patches_per_image,))
    Y = Y.reshape((patches_per_image,))
    C = np.stack((Y,X)).T
    
    idx = 0
    for j in range(len(C)):
        r,c = C[j]
        r,c = int(r), int(c)
        # check if gt patch has micronuclei or not
        gt_patch = gt[r:r+PS, c:c+PS]
        gt_labels = skimage.morphology.label(gt_patch)
        if np.max(gt_labels) > 0: # 1 represents only background
            # save the corresponding input and gt
            im_patch = im[r:r+PS, c:c+PS]
            
            # reconstruct filename
            imid = files[i].split('.')[0]
            suffix = f'.crop{idx}'
            new_imid = imid + suffix
            skimage.io.imsave(save_folder + new_imid + '.phenotype.tif', im_patch)
            skimage.io.imsave(save_folder + new_imid + '.phenotype_outlines.png', gt_labels.astype(np.uint16))
            idx = idx + 1

121


100%|██████████| 121/121 [00:04<00:00, 28.16it/s]


# Validation Data

In [82]:
path = '/scr/yren/micronuclei-detection/baselines/all_data_micronuclei_no_rescale/validation/'
save_folder = '/scr/yren/micronuclei-detection/baselines/microSAM_finetune_data/validation/'
files = os.listdir(path)
files = [file for file in files if file.endswith('.phenotype.tif')]
files = [file for file in files if not file.startswith('.')]
print(len(files))

PS = 256
for i in tqdm(range(len(files))):
    im = skimage.io.imread(path + files[i])
    gt = skimage.io.imread(path + files[i].replace('.phenotype.tif', '.phenotype_outlines.png'))

    H,W = im.shape
    patches_per_image = (W // PS) * (H // PS)
    X = np.linspace(0, W - W % PS, W // PS + 1)
    Y = np.linspace(0, H - H % PS, H // PS + 1)
    X,Y = np.meshgrid(X[:-1],Y[:-1], indexing='ij')
    X = X.reshape((patches_per_image,))
    Y = Y.reshape((patches_per_image,))
    C = np.stack((Y,X)).T
    for j in range(len(C)):
        r,c = C[j]
        r,c = int(r), int(c)
        # check if gt patch has micronuclei or not
        gt_patch = gt[r:r+PS, c:c+PS]
        gt_labels = skimage.morphology.label(gt_patch)
        
        idx = 0
        if np.max(gt_labels) > 0: # 1 represents only background
            # save the corresponding input and gt
            im_patch = im[r:r+PS, c:c+PS]
            
            # reconstruct filename
            imid = files[i].split('.')[0]
            suffix = f'.crop{idx}'
            new_imid = imid + suffix
            
            skimage.io.imsave(save_folder + new_imid + '.phenotype.tif', im_patch)
            skimage.io.imsave(save_folder + new_imid + '.phenotype_outlines.png', gt_labels.astype(np.uint16))
            idx = idx + 1

47


100%|██████████| 47/47 [00:01<00:00, 27.53it/s]


In [83]:
save_folder = '/scr/yren/micronuclei-detection/baselines/microSAM_finetune_data/train/'
files = os.listdir(save_folder)
files = [file for file in files if file.endswith('.phenotype_outlines.png')]
files = [file for file in files if not file.startswith('.')]

for i in range(len(files)):
    crop = skimage.io.imread(save_folder + files[i])
    labels = skimage.morphology.label(crop)
    if len(np.unique(labels)) == 1:
        print('Empty GT crop found!')

In [84]:
save_folder = '/scr/yren/micronuclei-detection/baselines/microSAM_finetune_data/validation/'
files = os.listdir(save_folder)
files = [file for file in files if file.endswith('.phenotype_outlines.png')]
files = [file for file in files if not file.startswith('.')]

for i in range(len(files)):
    crop = skimage.io.imread(save_folder + files[i])
    labels = skimage.morphology.label(crop)
    if len(np.unique(labels)) == 1:
        print('Empty GT crop found!')

In [75]:
import warnings
warnings.filterwarnings("ignore")

import os
from glob import glob
from IPython.display import FileLink
from typing import Union, Tuple, Optional

import numpy as np
import imageio.v3 as imageio
from matplotlib import pyplot as plt
from skimage.measure import label as connected_components

import torch

from torch_em.util.debug import check_loader
from torch_em.data import MinInstanceSampler
from torch_em.util.util import get_random_colors

import micro_sam.training as sam_training
from micro_sam.sample_data import fetch_tracking_example_data, fetch_tracking_segmentation_data
from micro_sam.automatic_segmentation import get_predictor_and_segmenter, automatic_instance_segmentation

- Leave ROI argument as None to use all data under the path

In [85]:
batch_size = 1  # the training batch size
patch_shape = (256, 256)  # the size of patches for training
sampler = MinInstanceSampler(min_size=25)

In [86]:
raw_key, label_key = '*.phenotype.tif', '*.phenotype_outlines.png'
train_dir = '/scr/yren/micronuclei-detection/baselines/microSAM_finetune_data/train/'
val_dir = '/scr/yren/micronuclei-detection/baselines/microSAM_finetune_data/validation/'
train_instance_segmentation = True
train_segmentation_dir = train_dir
val_segmentation_dir = val_dir

In [87]:
train_loader = sam_training.default_sam_loader(
    raw_paths=train_dir,
    raw_key=raw_key,
    label_paths=train_segmentation_dir,
    label_key=label_key,
    with_segmentation_decoder=train_instance_segmentation,
    patch_shape=patch_shape,
    batch_size=batch_size,
    is_seg_dataset=True,
    rois=None,
    shuffle=True,
    raw_transform=sam_training.identity,
    sampler=sampler
)

val_loader = sam_training.default_sam_loader(
    raw_paths=val_dir,
    raw_key=raw_key,
    label_paths=val_segmentation_dir,
    label_key=label_key,
    with_segmentation_decoder=train_instance_segmentation,
    patch_shape=patch_shape,
    batch_size=batch_size,
    is_seg_dataset=True,
    rois=None,
    shuffle=True,
    raw_transform=sam_training.identity,
    sampler=sampler
)

In [ ]:
n_objects_per_batch = 5  # the number of objects per batch that will be sampled
device = "cuda" if torch.cuda.is_available() else "cpu"  # the device/GPU used for training
n_epochs = 20  # how long we train (in epochs)

# The model_type determines which base model is used to initialize the weights that are finetuned.
# We use vit_b here because it can be trained faster. Note that vit_h usually yields higher quality results.
model_type = "vit_b_lm"

# The name of the checkpoint. The checkpoints will be stored in './checkpoints/<checkpoint_name>'
checkpoint_name = "sam_hela"

root_dir = '/scr/yren/micronuclei-detection/baselines/microSAM_finetune_data'

# Use tmux to run on workstation

In [ ]:
sam_training.train_sam(
    name=checkpoint_name,
    save_root=os.path.join(root_dir, "models"),
    model_type=model_type,
    train_loader=train_loader,
    val_loader=val_loader,
    n_epochs=n_epochs,
    n_objects_per_batch=n_objects_per_batch,
    with_segmentation_decoder=train_instance_segmentation,
    device=device
)

Verifying labels in 'val' dataloader: 100%|██████████| 50/50 [00:00<00:00, 62.09it/s]


Start fitting for 11180 iterations /  10 epochs
with 1118 iterations per epoch
Training with mixed precision


Epoch 0:   1%|          | 107/11180 [00:52<1:28:26,  2.09it/s]

KeyboardInterrupt: 